# clock/09 — *what* vs *when/where* dissociation: causal head-restriction test

**Question.** When we restrict a selected set of attention heads to look *only* at the
spec span of a **TARGET** step (> the CURRENT, prefilled step), does the model implement the
TARGET step instead? And which head-set drives this — **gaze** (*where*), **goalscope**
(*what*), or **decision** (*when*) heads — vs a matched **random** baseline?

**Setup.** `check_conditions(n)` over K predicate conditions, letters naming (`a,b,c,…`).
Priors (slots `< current`) are force-filled complete; the CURRENT slot is prefilled up to
`    {var} = ` and STOPPED — the **pivot**. The model must now choose the RHS.

**Two evals** (both):
- **Hard (oracle):** cut the generation at a linebreak, ask an oracle *(1 condition, 1 stub) →
  YES / NO / MALFORMED*, per condition. Success = YES(target) ∧ ¬YES(current).
- **Soft (likelihood):** teacher-force each condition's canonical RHS under the intervention,
  softmax the K log-likelihoods → `p_target`. Effect = `Δp_target = p_target(intv) − p_target(clean)`.

Records carry raw `current_slot` / `target_slot` so **forward vs backward** redirection,
slot-distance, and the full K-way confusion all fall out of the same table.

In [1]:
%load_ext autoreload
%autoreload 2

import os, sys, re, random, logging, json
sys.path.append("../..")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0,6")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
import numpy as np
import torch
from rich.console import Console
from rich.theme import Theme

from src.utils import logging_utils
logging_utils.configure(level=logging.INFO)
logger = logging.getLogger(__name__)
console = Console(theme=Theme({"default": "white"}), highlight=False)
def rich_print(*a, **k): console.print(*a, **k)

In [3]:
# ── model (heavy) ─────────────────────────────────────────────────────────────
from src.models import ModelandTokenizer

MODEL_NAME = "Qwen/Qwen2.5-Coder-32B-Instruct"
mt = ModelandTokenizer(
    model_key=MODEL_NAME, dtype=torch.bfloat16,
    device_map="auto", attn_implementation="eager",
)
print(f"n_layer={mt.n_layer}  n_embd={mt.n_embd}  "
      f"n_heads={mt.model.config.num_attention_heads}  "
      f"n_kv_heads={mt.model.config.num_key_value_heads}")

2026-07-14 14:58:59 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models
2026-07-14 14:58:59 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-14 14:58:59 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/config.json "HTTP/1.1 200 OK"
2026-07-14 14:58:59 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-14 14:58:59 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/tokenizer_config.json "HTTP/1.1 200 OK"
2026-07-14 14:58:59 ht

Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

2026-07-14 14:59:11 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-14 14:59:11 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/generation_config.json "HTTP/1.1 200 OK"
2026-07-14 14:59:11 src.models INFO     loaded model <Qwen/Qwen2.5-Coder-32B-Instruct> | size: 62492.135 MB | dtype: torch.bfloat16 | device: cuda:0
n_layer=64  n_embd=5120  n_heads=40  n_kv_heads=8


## 1. Dissociation prompt builder

Reuse `build_example(style="singleline")` (priors complete, distinct predicates — dedup is
guaranteed by `rng.sample` without replacement) and **truncate** the target line at its
assignment `=`, leaving `    {var} = ` as the pivot. Validated offline on the Qwen tokenizer.

In [4]:
from src.goals.multiline_prompts import build_example

def build_dissociation_prompt(mt, rng, current_slot, n_conditions,
                              naming="letters", lang="python", **kw):
    """Priors (< current_slot) complete AND single-line; CURRENT slot prefilled up to `    {var} =`
    then STOP. All priors are single-line (prior_style="singleline") so a multi-line prior can't
    nudge the target var toward a multi-line implementation. The pivot (final ` =` token) is where
    the model chooses the RHS. Returns build_example dict + dissoc_prompt, current_slot, pivot_pos,
    prompt_ids."""
    ex = build_example(mt=mt, rng=rng, target_slot=current_slot, n_conditions=n_conditions,
                       mode="chat", style="singleline", naming=naming, lang=lang,
                       prior_style="singleline", **kw)
    tvar = ex["target_var"]
    fp = ex["full_prompt"]
    line_start = fp.rfind(f"\n    {tvar} =") + 1        # target var letter appears only on its line
    assign_eq  = fp.index("=", line_start)              # assignment '=' precedes any ==,>=,<=
    cut = assign_eq + 1                                 # end exactly at '=' (no dangling space)
    ex["dissoc_prompt"] = fp[:cut]
    ex["current_slot"]  = current_slot
    ids = mt.tokenizer(ex["dissoc_prompt"], add_special_tokens=False)["input_ids"]
    ex["prompt_ids"] = ids
    ex["pivot_pos"]  = len(ids) - 1
    return ex

In [5]:
# demo — 2 samples
for seed in (7, 21):
    ex = build_dissociation_prompt(mt, random.Random(seed), current_slot=2, n_conditions=5)
    print("=" * 70)
    print(f"seed={seed}  CURRENT={ex['current_slot']} (var {ex['target_var']}, {ex['target_pred']})"
          f"  K={ex['n_conditions']}  pivot_pos={ex['pivot_pos']}")
    for i, t in enumerate(ex["spec"].spec_texts):
        mark = "  <== CURRENT" if i == ex["current_slot"] else ""
        print(f"   [{i}] {ex['var_names'][i]}: {t}{mark}")
    print("   code tail:", repr(ex["dissoc_prompt"].split("<code>\n")[-1][-80:]))
    print("   last tokens:", [mt.tokenizer.decode([t]) for t in ex["prompt_ids"][-5:]])

seed=7  CURRENT=2 (var c, digit_sum_gt)  K=5  pivot_pos=288
   [0] a: n is a power of two
   [1] b: n is a perfect cube
   [2] c: the sum of n's digits is greater than 5  <== CURRENT
   [3] d: n is odd
   [4] e: all of n's digits are even
   code tail: '0 and (n & (n - 1)) == 0\n    b = round(abs(n) ** (1 / 3)) ** 3 == abs(n)\n    c ='
   last tokens: ['(n', ')\n', '   ', ' c', ' =']
seed=21  CURRENT=2 (var c, abundant)  K=5  pivot_pos=320
   [0] a: n is a triangular number
   [1] b: the sum of n's digits is divisible by 4
   [2] c: the sum of n's proper divisors is greater than n  <== CURRENT
   [3] d: n has more than 4 ones in its binary representation
   [4] e: n is smaller than the number obtained by reversing its digits
   code tail: '0.5) ** 2 == 8 * n + 1\n    b = sum(int(d) for d in str(abs(n))) % 4 == 0\n    c ='
   last tokens: ['0', '\n', '   ', ' c', ' =']


## 2. Locate condition spec spans — *where* to restrict attention

Per condition k: the token range of its full spec text in the prompt (+ id_substr sub-ranges).
`restrict_span_positions` picks which key positions the selected heads may keep looking at
(full spec text by default; switch to id_substrs for a tighter span). Validated offline.

In [6]:
from src.tokens import find_token_range

def locate_condition_spans(mt, ex):
    """Per condition k → {cond, var, text, full_span [s,e], id_substr_spans [[s,e],...]},
    token ranges into the `dissoc_prompt` tokenization (add_special_tokens=False)."""
    prompt = ex["dissoc_prompt"]
    offsets = [tuple(p) for p in mt.tokenizer(
        prompt, return_offsets_mapping=True, add_special_tokens=False)["offset_mapping"]]
    spec = ex["spec"]
    out = []
    for k, (text, subs) in enumerate(zip(spec.spec_texts, spec.id_substrs)):
        base = prompt.find(text)
        full = None
        if base != -1:
            try:
                full = list(find_token_range(prompt, text, offset_mapping=offsets, occurrence=0))
            except (ValueError, AssertionError):
                full = None
        sub_ranges = []
        if base != -1:
            for ss in subs:
                rel = text.find(ss)
                if rel == -1:
                    continue
                occ = prompt.count(ss, 0, base + rel)
                try:
                    ts, te = find_token_range(prompt, ss, offset_mapping=offsets, occurrence=occ)
                    sub_ranges.append([ts, te])
                except (ValueError, AssertionError):
                    continue
        out.append({"cond": k, "var": ex["var_names"][k], "text": text,
                    "full_span": full, "id_substr_spans": sub_ranges})
    return out

def restrict_span_positions(spans, slot, which="full"):
    """Key positions the restricted heads may attend to for a given condition slot.
    which='full' → the whole spec-text span; 'id' → only id_substr tokens."""
    s = spans[slot]
    if which == "id":
        return sorted({p for r in s["id_substr_spans"] for p in range(r[0], r[1])})
    lo, hi = s["full_span"]
    return list(range(lo, hi))

In [7]:
# demo
ex = build_dissociation_prompt(mt, random.Random(7), current_slot=2, n_conditions=5)
spans = locate_condition_spans(mt, ex)
toks = [mt.tokenizer.decode([t]) for t in ex["prompt_ids"]]
for s in spans:
    ft = "".join(toks[s["full_span"][0]:s["full_span"][1]]) if s["full_span"] else None
    print(f"[{s['cond']}] {s['var']}  full_span={s['full_span']}  ->{ft!r}")
print("keep-positions for TARGET=4 (full):", restrict_span_positions(spans, 4, "full"))
print("keep-positions for TARGET=4 (id)  :", restrict_span_positions(spans, 4, "id"))

[0] a  full_span=[70, 76]  ->' n is a power of two'
[1] b  full_span=[79, 84]  ->' n is a perfect cube'
[2] c  full_span=[87, 98]  ->" the sum of n's digits is greater than 5"
[3] d  full_span=[101, 104]  ->' n is odd'
[4] e  full_span=[107, 114]  ->" all of n's digits are even"
keep-positions for TARGET=4 (full): [107, 108, 109, 110, 111, 112, 113]
keep-positions for TARGET=4 (id)  : [107, 111, 113]


## 3. Canonical references + RHS scoring targets — *soft-eval* candidates

Per condition: the canonical single-line impl, and the RHS after stripping `    {var} = `
(mechanical — no oracle). These RHS token ids are what the soft eval teacher-forces. One
canonical form per condition (M=1) so no condition is inflated by having more variants.
Validated offline.

In [8]:
from src.goals.condition_data import INT_PREDICATES
_PRED_BY_ID = {p.id: p for p in INT_PREDICATES}

def get_condition_references(mt, ex):
    """Per condition k → {cond, var, pred, has_singleline, full_line, rhs, rhs_ids, rhs_toks}.
    `rhs` KEEPS the space after `=` (leading space) so that prompt_ids (ending at the ` =`
    token) + rhs_ids tokenizes identically to the natural `    {var} = {expr}` line."""
    spec = ex["spec"]
    out = []
    for k in range(ex["n_conditions"]):
        pid, prm, var = spec.predicate_ids[k], spec.predicate_params[k], ex["var_names"][k]
        pred = _PRED_BY_ID[pid]
        has_sl = pred.has_singleline_for(ex["lang"])
        full_line = (pred.render_singleline(prm, var, lang=ex["lang"]) if has_sl
                     else " / ".join(pred.render_multiline(prm, var, lang=ex["lang"])))
        m = re.match(rf"^\s*{re.escape(var)}\s*=", full_line)   # strip `{var} =`, KEEP the space
        rhs = full_line[m.end():] if m else full_line
        rhs_ids = mt.tokenizer(rhs, add_special_tokens=False)["input_ids"]
        out.append({"cond": k, "var": var, "pred": pid, "has_singleline": has_sl,
                    "full_line": full_line, "rhs": rhs, "rhs_ids": rhs_ids,
                    "rhs_toks": [mt.tokenizer.decode([t]) for t in rhs_ids]})
    return out

In [9]:
# demo
refs = get_condition_references(mt, ex)
for r in refs:
    flag = "" if r["has_singleline"] else "  [NO SINGLELINE]"
    print(f"[{r['cond']}] {r['var']} ({r['pred']}){flag}  rhs={r['rhs']!r}")
if not all(r["has_singleline"] for r in refs):
    print("WARNING: some conditions lack a single-line form; constrain the predicate pool.")

[0] a (power_of_two)  rhs=' n > 0 and (n & (n - 1)) == 0'
[1] b (perfect_cube)  rhs=' round(abs(n) ** (1 / 3)) ** 3 == abs(n)'
[2] c (digit_sum_gt)  rhs=' sum(int(d) for d in str(abs(n))) > 5'
[3] d (odd)  rhs=' n % 2 == 1'
[4] e (all_digits_even)  rhs=' all(int(c) % 2 == 0 for c in str(abs(n)))'


## 4. Head sets — gaze (*where*) / goalscope (*what*) / decision (*when*) / random

- **gaze**: top-K step-lookback heads (`score_condition_heads.load_head_scores`).
- **goalscope**: trained goal-head mask (`results/localization/<model>/head_mask.pt`).
- **decision**: completion mask (`train_completion_mask.py`, tag `lam10.0`).
- **random**: matched **count** *and* **layer band** to the head-set under test (fair baseline).

In [10]:
from src.goals.head_loc_attn import HeadMask
from src.utils import env_utils
from scripts.clock.score_condition_heads import load_head_scores

N_LAYER = mt.n_layer
N_HEADS = mt.model.config.num_attention_heads

# -- gaze heads (top-K step-lookback) --
GAZE_TOPK = 150
gaze_scores = load_head_scores(model=MODEL_NAME, version="conditions", q_span=15,
                               compare_with="mean")["combined"]         # (n_layers, n_heads)
flat = np.argsort(gaze_scores, axis=None)[::-1][:GAZE_TOPK]
gaze_heads = [(int(i // N_HEADS), int(i % N_HEADS)) for i in flat]

# -- goalscope heads (trained mask) --
GOAL_THRESHOLD = 0.9
ck = torch.load(os.path.join(env_utils.DEFAULT_RESULTS_DIR,
                f"localization/{MODEL_NAME.split('/')[-1]}/head_mask.pt"), map_location="cpu")
goal_layers = [int(x) for x in ck["layers"]]
gmask = ck["mask_values"].numpy()
if mt.name.split("/")[-1] == "Qwen2.5-Coder-32B-Instruct":
    gmask[0:25] = 0  # zero out early layers
    gmask[56:] = 0  # zero out late layers
elif mt.name.split("/")[-1] == "Llama-3.3-70B-Instruct":
    gmask[0:18] = 0  # zero out early layers
    gmask[45:] = 0  # zero out late layers

goalscope_heads = [(goal_layers[li], int(h))
                   for li, h in np.argwhere(gmask > GOAL_THRESHOLD)]

def random_heads_like(reference_heads, rng, n=None):
    """Random heads matched on count and per-layer distribution of `reference_heads`."""
    from collections import Counter
    per_layer = Counter(l for l, _ in reference_heads)
    out = []
    for l, cnt in per_layer.items():
        choices = rng.sample(range(N_HEADS), min(cnt, N_HEADS))
        out += [(l, h) for h in choices]
    return out

rng_heads = random.Random(0)
HEAD_SETS = {
    "gaze":      gaze_heads,
    "goalscope": goalscope_heads,
    "random":    random_heads_like(gaze_heads, rng_heads),   # matched to gaze by default
}
for name, hs in HEAD_SETS.items():
    layers = sorted({l for l, _ in hs})
    print(f"{name:10s}: {len(hs):4d} heads  ·  layers {min(layers)}–{max(layers)}")
print(f"gaze∩goalscope Jaccard = "
      f"{len(set(gaze_heads)&set(goalscope_heads))/len(set(gaze_heads)|set(goalscope_heads)):.3f}")

gaze      :  150 heads  ·  layers 8–61
goalscope :  122 heads  ·  layers 28–55
random    :  150 heads  ·  layers 8–61
gaze∩goalscope Jaccard = 0.075


## 5. Core intervention — restrict selected heads to the TARGET span

Vectorised attention restriction (mirrors `forward_with_attn_knockout` from `06_loc_head.ipynb`,
same nnsight node path `attention_interface_0.source.nn_functional_softmax_0`): for each selected
`(layer, head)`, set pre-softmax scores from every **query position in `q_positions`** to
`-inf` for all keys **except** `keep_positions` (the TARGET spec span). Other heads are untouched.

> Needs the loaded model. Structure follows the validated knockout util; run and eyeball the
> saved attention pattern for one head to confirm the mask lands on the target span.

In [11]:
from src.functional import get_module_nnsight

@torch.no_grad()
def forward_restrict_to_span(mt, inputs, selected_heads, keep_positions, q_positions,
                             save_pattern_for=None):
    """Forward pass where each (layer, head) in `selected_heads` may attend, from each query in
    `q_positions`, ONLY to `keep_positions` (all other keys → -inf pre-softmax).
    Returns (logits (batch, seq, vocab) float32, {'attn_patterns': {(l,h): (seq,seq)}})."""
    seq_len = inputs["input_ids"].shape[1]
    device  = next(mt._model.parameters()).device
    keep = torch.tensor(sorted(set(keep_positions)), device=device, dtype=torch.long)
    qpos = torch.tensor(sorted(set(q_positions)),   device=device, dtype=torch.long)

    # per-layer additive mask (1, n_heads, seq, seq): -inf on (head, q, ~keep)
    by_layer = {}
    for (l, h) in selected_heads:
        m = by_layer.get(l)
        if m is None:
            m = torch.zeros(1, N_HEADS, seq_len, seq_len, device=device)
            by_layer[l] = m
        m[0, h, qpos.unsqueeze(1), :] = -torch.inf     # cut all keys from these queries
        m[0, h, qpos.unsqueeze(1), keep.unsqueeze(0)] = 0.0   # ...restore the target span

    save_by_layer = {}
    if save_pattern_for:
        for l, h in save_pattern_for:
            save_by_layer.setdefault(l, []).append(h)

    proxies = {}
    with mt.trace(inputs, scan=False, validate=False):
        for l in set(by_layer) | set(save_by_layer):
            ai = get_module_nnsight(mt, mt.attn_module_name_format.format(l)).source.attention_interface_0
            if l in by_layer:
                sm = ai.source.nn_functional_softmax_0
                sm.input = sm.input + by_layer[l].to(sm.input.device)
            if l in save_by_layer:
                proxies[l] = ai.source.to_0.output.save()
        logits = mt.output.logits.save()

    patterns = {}
    for l, heads in save_by_layer.items():
        full = proxies[l]
        for h in heads:
            patterns[(l, h)] = full[0, h]
    return logits.float(), {"attn_patterns": patterns}

In [17]:
from collections import defaultdict

@torch.no_grad()
def generate_restricted_to_span(mt, prompt_ids, selected_heads, keep_positions, q_start,
                                max_new=30, stop_suffix=None):
    """Greedy, NON-CACHED decode: the full sequence is recomputed each step and the attention
    restriction is re-applied to ALL query rows >= q_start of each `selected_heads` (they may
    attend ONLY to `keep_positions`; all other keys, incl. newly generated ones, get -inf).
    Because it recomputes (no KV cache), the restriction stays active for every generated token —
    the pivot AND each RHS token — and the modified representations propagate to later tokens via
    causal attention. `selected_heads=[]` -> baseline. Mirrors `generate_with_head_attn_mask`
    from 02_condition_gaze.ipynb.

    Stops once a generated token's decoded text ends with any string in `stop_suffix`
    (default: [eos, "\\n", "\\n\\n"]); eos is detected by id (not appended). Returns the decoded
    string with trailing newlines stripped."""
    if stop_suffix is None:
        stop_suffix = [mt.tokenizer.eos_token, "\n", "\n\n"]
    stop_suffix = [s for s in stop_suffix if s]                # drop None/empty
    device = next(mt._model.parameters()).device
    ids = torch.tensor([list(prompt_ids)], device=device)
    l2h = defaultdict(list)
    for l, h in sorted(selected_heads):
        l2h[l].append(h)
    keep_set = set(int(k) for k in keep_positions)
    gen = []
    for _ in range(max_new):
        seq = ids.shape[1]
        kvec = torch.zeros(seq, device=device)
        if selected_heads:
            out_keys = [k for k in range(seq) if k not in keep_set]   # mask all but the span
            if out_keys:
                kvec[torch.tensor(out_keys, device=device)] = float("-inf")
        with mt.trace({"input_ids": ids, "attention_mask": torch.ones_like(ids)},
                      scan=False, validate=False):
            for L in sorted(l2h):                       # ascending layer = nnsight exec order
                sm = get_module_nnsight(mt, mt.attn_module_name_format.format(L)) \
                        .source.attention_interface_0.source.nn_functional_softmax_0
                km = torch.zeros(1, N_HEADS, seq, seq, device=device)
                for h in l2h[L]:
                    km[0, h, q_start:, :] += kvec
                sm.input = sm.input + km.to(sm.input.device, sm.input.dtype)
            logits = mt.output.logits[:, -1, :].save()
        nxt = int(logits[0].argmax(-1))
        if nxt == mt.tokenizer.eos_token_id:            # eos: stop, don't append
            break
        tok_str = mt.tokenizer.decode([nxt])
        gen.append(nxt)
        if any(tok_str.endswith(s) for s in stop_suffix):   # token ends a line -> stop
            break
        ids = torch.cat([ids, torch.tensor([[nxt]], device=device)], dim=1)
    return mt.tokenizer.decode(gen, skip_special_tokens=True)

## 6. Soft eval (Approach 2) — per-condition RHS log-likelihood

For a fixed intervention (restrict `selected_heads` to the TARGET span), teacher-force each
candidate condition's RHS from the pivot and sum its token log-probs. Softmax over the K
LLs → closed-set `p_k`. `soft_metrics` (pure) reports `p_target`, `p_current`, the target/current
margin, argmax and TARGET rank. Compare intervention vs a clean (no-restrict) pass.

In [36]:
@torch.no_grad()
def score_condition_lls(mt, ex, refs, selected_heads=None, keep_positions=None, length_norm=True):
    """(K,) summed log p of each condition's canonical RHS, teacher-forced from the pivot.
    If selected_heads/keep_positions are given, the restriction is applied to the RHS query
    positions (pivot .. pivot+len(rhs)-1); else a clean pass. One forward per candidate.

    length_norm=True (default) returns the MEAN per-token log-prob (sum / n_rhs_tokens), so a
    candidate is not penalised just for being longer; length_norm=False returns the raw summed
    log-likelihood (biased toward shorter RHS)."""
    from src.tokens import prepare_input
    pivot = ex["pivot_pos"]
    base_ids = list(ex["prompt_ids"])
    lls = []
    for r in refs:
        rhs_ids = r["rhs_ids"]
        ids = base_ids + rhs_ids
        inp = {"input_ids": torch.tensor([ids], device=next(mt._model.parameters()).device),
               "attention_mask": torch.ones(1, len(ids), device=next(mt._model.parameters()).device)}
        if selected_heads and keep_positions is not None:
            qpos = list(range(pivot, pivot + len(rhs_ids)))     # queries that emit the RHS
            logits, _ = forward_restrict_to_span(mt, inp, selected_heads, keep_positions, qpos)
        else:
            with mt.trace(inp, scan=False, validate=False):
                logits = mt.output.logits.save()
            logits = logits.float()
        logp = torch.log_softmax(logits[0], dim=-1)             # (seq, vocab)
        # token t of rhs is predicted at position pivot+t (0-indexed), target id rhs_ids[t]
        tgt = torch.tensor(rhs_ids)
        pos = torch.arange(pivot, pivot + len(rhs_ids))
        s = logp[pos, tgt].sum()
        lls.append(float((s / len(rhs_ids) if length_norm else s).cpu()))
    return np.asarray(lls)


def soft_metrics(cond_lls, current, target, tau=1.0):
    lls = np.asarray(cond_lls, dtype=np.float64)
    p = np.exp((lls - lls.max()) / tau); p = p / p.sum()
    order = np.argsort(-lls)
    rank_target = int(np.where(order == target)[0][0]) + 1
    return {"p": p, "p_target": float(p[target]), "p_current": float(p[current]),
            "margin_tgt_cur": float(lls[target] - lls[current]),
            "argmax": int(order[0]), "rank_target": rank_target,
            "mrr_target": 1.0 / rank_target, "redirected_argmax": bool(order[0] == target)}

In [37]:
HEAD_SETS.keys()

dict_keys(['gaze', 'goalscope', 'random'])

In [39]:
# demo — one (current, target) pair, gaze restriction vs clean
from rich.table import Table
from rich.panel import Panel
from rich.markup import escape
from rich.console import Group

CURRENT, TARGET, K = 2, 4, 5
HEADSET = "goalscope"   # "gaze" | "goalscope" | "random"
WHICH   = "full"        # "full" spec span | "id" substrs only
MAX_NEW = 64
CUR_C, TGT_C = "sea_green1", "bright_cyan"            # CURRENT / TARGET highlight colors

ex    = build_dissociation_prompt(mt, random.Random(1123), current_slot=CURRENT, n_conditions=K)
spans = locate_condition_spans(mt, ex)
refs  = get_condition_references(mt, ex)
keep  = restrict_span_positions(spans, TARGET, which=WHICH)
toks  = [mt.tokenizer.decode([t]) for t in ex["prompt_ids"]]
sel   = HEAD_SETS[HEADSET]
cur = tgt = ex["spec"]                                # alias for readability
_dir = "FORWARD (skip ahead)" if TARGET > CURRENT else "BACKWARD (go back)"

# ── BLOCK 1 — full prompt + what we're intervening on (one grouped block) ──────
rich_print(Group(
    Panel(f"\"{escape(mt.tokenizer.decode(ex['prompt_ids']))}\"",
          title="[bold]full prompt[/bold]", border_style="grey42"),
    Panel.fit(ex["dissoc_prompt"].split("<code>\n")[-1].rstrip(),
              title="[bold]assistant code — ends at pivot `{var} =`[/bold]", border_style="cyan"),
    f"[bold {CUR_C}]CURRENT_COND[/] = slot {CURRENT}  var=[{CUR_C}]{ex['var_names'][CURRENT]}[/]"
    f"  pred=[{CUR_C}]{cur.predicate_ids[CURRENT]}[/]  “{cur.spec_texts[CURRENT]}”",
    f"[bold {TGT_C}]TARGET_COND [/] = slot {TARGET}  var=[{TGT_C}]{ex['var_names'][TARGET]}[/]"
    f"  pred=[{TGT_C}]{tgt.predicate_ids[TARGET]}[/]  “{tgt.spec_texts[TARGET]}”",
    f"direction   = [bold]{_dir}[/]   |Δslot|={abs(TARGET-CURRENT)}   pivot_pos={ex['pivot_pos']}",
    f"CURRENT rhs = [{CUR_C}]{refs[CURRENT]['rhs']!r}[/]   ({len(refs[CURRENT]['rhs_ids'])} tok)",
    f"TARGET  rhs = [{TGT_C}]{refs[TARGET]['rhs']!r}[/]   ({len(refs[TARGET]['rhs_ids'])} tok)",
    f"restrict [bold]{HEADSET}[/] heads (n={len(sel)}) so their attn keys ⊆ TARGET span "
    f"[{min(keep)}..{max(keep)}] = [{TGT_C}]{''.join(toks[min(keep):max(keep)+1])!r}[/]",
))

# ── RUN: clean vs intervention (SOFT: teacher-forced RHS log-likelihood) ───────
ll_clean = score_condition_lls(mt, ex, refs)
ll_intv  = score_condition_lls(mt, ex, refs, selected_heads=sel, keep_positions=keep)
m_clean  = soft_metrics(ll_clean, CURRENT, TARGET)
m_intv   = soft_metrics(ll_intv,  CURRENT, TARGET)

# ── BLOCK 2 — per-condition LL/p table + soft-metric summary (one grouped block) ─
tbl = Table(title=f"per-condition RHS log-likelihood & closed-set p  ({HEADSET}→TARGET, {WHICH})")
tbl.add_column("slot"); tbl.add_column("var"); tbl.add_column("pred")
tbl.add_column("rhs", overflow="fold", max_width=32)
tbl.add_column("LL/tok cln", justify="right"); tbl.add_column("p clean", justify="right")
tbl.add_column("LL/tok int", justify="right");  tbl.add_column("p intv", justify="right")
tbl.add_column("Δp", justify="right")
for k in range(K):
    tag   = "CUR" if k == CURRENT else ("TGT" if k == TARGET else "")
    style = CUR_C if k == CURRENT else (TGT_C if k == TARGET else None)
    dp = m_intv["p"][k] - m_clean["p"][k]
    tbl.add_row(f"{k} {tag}", ex["var_names"][k], refs[k]["pred"], refs[k]["rhs"],
                f"{ll_clean[k]:+.2f}", f"{m_clean['p'][k]:.3f}",
                f"{ll_intv[k]:+.2f}",  f"{m_intv['p'][k]:.3f}", f"{dp:+.3f}", style=style)

def _mline(name, m):
    return (f"[bold]{name:16s}[/] argmax=slot {m['argmax']}  p_target={m['p_target']:.3f}  "
            f"p_current={m['p_current']:.3f}  rank_target={m['rank_target']}  "
            f"redirected={m['redirected_argmax']}")
rich_print(Group(
    tbl,
    _mline("CLEAN", m_clean),
    _mline(f"{HEADSET}→TARGET", m_intv),
    f"[bold green]Δp_target = {m_intv['p_target']-m_clean['p_target']:+.3f}[/]   "
    f"Δmargin(tgt-cur) = {m_intv['margin_tgt_cur']-m_clean['margin_tgt_cur']:+.2f}   "
    f"(clean rank_t={m_clean['rank_target']} → intv rank_t={m_intv['rank_target']})",
))

# ── BLOCK 3 — what does the LM actually GENERATE under the restriction? ────────
# greedy, non-cached, mask re-applied each step; cut at newline. Control = random heads matched
# to `sel` on count + layer band (fair baseline). Same as 02_condition_gaze §4.
q_start   = ex["pivot_pos"]
ctrl_sel  = random_heads_like(sel, random.Random(0))
tvar      = ex["var_names"][CURRENT]
base  = generate_restricted_to_span(mt, ex["prompt_ids"], [],       keep, q_start, MAX_NEW)
restr = generate_restricted_to_span(mt, ex["prompt_ids"], sel,      keep, q_start, MAX_NEW)
ctrl  = generate_restricted_to_span(mt, ex["prompt_ids"], ctrl_sel, keep, q_start, MAX_NEW)
rich_print(Group(
    f"[bold]generated `{tvar} =` (greedy, cut@newline):[/]  "
    f"CURRENT=slot{CURRENT} [{CUR_C}]{cur.predicate_ids[CURRENT]}[/]  "
    f"TARGET=slot{TARGET} [{TGT_C}]{tgt.predicate_ids[TARGET]}[/]",
    f"  baseline   no mask             {tvar} =[sea_green1]{escape(base)!r}[/]",
    f"  restrict   {HEADSET}⊂cond{TARGET}   {tvar} =[salmon1]{escape(restr)!r}[/]",
    f"  control    {len(ctrl_sel)} random heads    {tvar} =[plum1]{escape(ctrl)!r}[/]",
))

╭────────────────────────────────────────────────── full prompt ──────────────────────────────────────────────────╮
│ "<|im_start|>system                                                                                             │
│ You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical   │
│ correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>                              │
│ <|im_start|>user                                                                                                │
│ Write a Python program that implements a function `check_conditions` which, given an integer n, checks the      │
│ following conditions:                                                                                           │
│                                                                                                                 │
│ 1. n has at least 4 digits                                                                                      │
│ 2. n has more than 1 ones in its binary representation                                                          │
│ 3. n has more than 6 divisors                                                                                   │
│ 4. n is even                                                                                                    │
│ 5. the sum of n's digits is greater than 15                                                                     │
│                                                                                                                 │
│ For each condition, in the order listed, assign the boolean result of the check to its own local variable,      │
│ using single lowercase letters in order — `a` for the first condition, `b` for the second, and so on. After all │
│ conditions have been checked, use a SINGLE print statement that, using those variables, reports whether each    │
│ condition passed individually and whether all of them passed collectively.                                      │
│ Compute each condition's boolean result inside `check_conditions` (one local variable per condition); do not    │
│ define any helper functions.                                                                                    │
│ Do not include any comments in your code.<|im_end|>                                                             │
│ <|im_start|>assistant                                                                                           │
│ <code>                                                                                                          │
│ def check_conditions(n: int):                                                                                   │
│     a = len(str(abs(n))) >= 4                                                                                   │
│     b = bin(abs(n)).count("1") > 1                                                                              │
│     c ="                                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─ assistant code — ends at pivot `{var} =` ─╮
│ def check_conditions(n: int):              │
│     a = len(str(abs(n))) >= 4              │
│     b = bin(abs(n)).count("1") > 1         │
│     c =                                    │
╰────────────────────────────────────────────╯
CURRENT_COND = slot 2  var=c  pred=num_divisors_gt  “n has more than 6 divisors”
TARGET_COND  = slot 4  var=e  pred=digit_sum_gt  “the sum of n's digits is greater than 15”
direction   = FORWARD (skip ahead)   |Δslot|=2   pivot_pos=282
CURRENT rhs = ' sum(1 for d in range(1, abs(n) + 1) if n % d == 0) > 6'   (28 tok)
TARGET  rhs = ' sum(int(d) for d in str(abs(n))) > 15'   (15 tok)
restrict goalscope heads (n=122) so their attn keys ⊆ TARGET span [111..122] = " the 

                     per-condition RHS log-likelihood & closed-set p  (goalscope→TARGET, full)                     
┏━━━━━━━┳━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┓
┃ slot  ┃ var ┃ pred            ┃ rhs                       ┃ LL/tok cln ┃ p clean ┃ LL/tok int ┃ p intv ┃     Δp ┃
┡━━━━━━━╇━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━┩
│ 0     │ a   │ num_digits_ge   │  len(str(abs(n))) >= 4    │      -3.75 │   0.015 │      -1.74 │  0.130 │ +0.115 │
│ 1     │ b   │ set_bits_gt     │  bin(abs(n)).count("1") > │      -1.73 │   0.114 │      -1.81 │  0.121 │ +0.007 │
│       │     │                 │ 1                         │            │         │            │        │        │
│ 2 CUR │ c   │ num_divisors_gt │  sum(1 for d in range(1,  │      -0.24 │   0.509 │      -1.96 │  0.104 │ -0.405 │
│       │     │                 │ abs(n) + 1) if n % d ==   │            │         │            │        │        │
│       │     │                 │ 0) > 6                    │            │         │            │        │        │
│ 3     │ d   │ even            │  n % 2 == 0               │      -3.05 │   0.031 │      -3.39 │  0.025 │ -0.006 │
│ 4 TGT │ e   │ digit_sum_gt    │  sum(int(d) for d in      │      -0.67 │   0.331 │      -0.18 │  0.619 │ +0.288 │
│       │     │                 │ str(abs(n))) > 15         │            │         │            │        │        │
└───────┴─────┴─────────────────┴───────────────────────────┴────────────┴─────────┴────────────┴────────┴────────┘
CLEAN            argmax=slot 2  p_target=0.331  p_current=0.509  rank_target=2  redirected=False
goalscope→TARGET argmax=slot 4  p_target=0.619  p_current=0.104  rank_target=1  redirected=True
Δp_target = +0.288   Δmargin(tgt-cur) = +2.21   (clean rank_t=2 → intv rank_t=1)

generated `c =` (greedy, cut@newline):  CURRENT=slot2 num_divisors_gt  TARGET=slot4 digit_sum_gt
  baseline   no mask             c =' sum(1 for i in range(1, abs(n) + 1) if abs(n) % i == 0) > 6\n'
  restrict   goalscope⊂cond4   c =' sum(int(digit) for digit in str(abs(n))) > 15\n'
  control    122 random heads    c =' len(\) > 6\n'

## 7. Hard eval (Approach 1) — oracle judge

Generate under the intervention, cut at the first linebreak, then ask an oracle *per condition*
*(1 condition, 1 stub) → YES / NO / MALFORMED*. Judge has **no** notion of current/target — just
"does this stub attempt THIS condition?" MALFORMED grace: buggy-but-conceptual → YES; degenerate
→ MALFORMED (few-shot in the system prompt). Prompt builder + parser validated offline; wire the
oracle call to the project's LLM client. Headline: **flip = YES(target) ∧ ¬YES(current)**.

In [40]:
import hashlib
from src.utils.oracle_llms import ask_claude

ORACLE_SYS = (
    "You are a meticulous code reviewer. You are shown ONE natural-language condition and ONE "
    "short Python code stub (the right-hand side of an assignment, possibly incomplete or slightly "
    "malformed). Decide whether the stub is a genuine attempt to implement THAT condition.\n\n"
    "Answer with exactly one token:\n"
    "  YES        — the stub implements (or clearly attempts) the given condition, EVEN IF it has\n"
    "               minor bugs or would not compile (e.g. `text == text.reversed()` for a palindrome\n"
    "               check: `.reversed()` is not valid Python, but the INTENT is unmistakable -> YES).\n"
    "  NO         — the stub is well-formed / meaningful code but implements a DIFFERENT condition.\n"
    "  MALFORMED  — the stub is degenerate and expresses no coherent check at all\n"
    "               (e.g. `= = = =`, `s == s == d == e ==`, empty, or pure repetition, or just garbage).\n\n"
    "Judge the CONCEPT. Output only YES, NO, or MALFORMED."
)
ORACLE_MODEL = "claude-sonnet-4-6"
# ask_claude hashes its cache on (prompt, temp, max_tokens) but NOT the system prompt, so we fold a
# short digest of ORACLE_SYS into the user prompt -> editing the rubric correctly busts the cache.
_SYS_TAG = hashlib.md5(ORACLE_SYS.encode()).hexdigest()[:8]


def build_oracle_prompt(condition_text, code_stub):
    """User-message content for one (condition, stub) judgement (rubric-tagged for cache safety)."""
    return (f"[rubric {_SYS_TAG}]\n"
            f"Condition: {condition_text}\nCode stub: {code_stub}\nAnswer:")


def parse_oracle(resp):
    r = resp.strip().upper()
    for tag in ("MALFORMED", "YES", "NO"):
        if r.startswith(tag):
            return tag
    return "UNPARSED"


def judge_condition(condition_text, code_stub, model_name=ORACLE_MODEL, use_cache=True):
    """Oracle verdict for whether `code_stub` implements `condition_text`: YES | NO | MALFORMED
    (| UNPARSED). Deterministic (temperature=0), cached via src.utils.oracle_llms.ask_claude."""
    resp = ask_claude(
        prompt=build_oracle_prompt(condition_text, code_stub),
        system_prompt=ORACLE_SYS,
        temperature=0.0, max_tokens=8, use_cache=use_cache, model_name=model_name,
    )
    return parse_oracle(resp)


def judge_stub_kway(ex, stub, use_cache=True):
    """Judge `stub` against ALL K conditions of `ex` (the oracle has NO current/target notion).
    Returns list[str] of per-slot verdicts (YES | NO | MALFORMED | UNPARSED)."""
    return [judge_condition(ex["spec"].spec_texts[k], stub, use_cache=use_cache)
            for k in range(ex["n_conditions"])]


def classify_stub(verdicts, current, target):
    """Outcome of a per-slot verdict vector relative to CURRENT/TARGET → (label, yes_slots).
    The headline redirect (flip) = YES(target) ∧ ¬YES(current)."""
    yes = [k for k, v in enumerate(verdicts) if v == "YES"]
    n_mal = sum(v == "MALFORMED" for v in verdicts)
    if not yes and n_mal >= len(verdicts) / 2:
        return "MALFORMED", yes
    if target in yes and current not in yes:
        return "TARGET (flip)", yes
    if current in yes and target not in yes:
        return "CURRENT (no change)", yes
    if current in yes and target in yes:
        return "AMBIGUOUS (both)", yes
    if yes:
        return f"OTHER slot(s) {yes}", yes
    return "NONE", yes


# demo — real oracle calls (3 canonical cases: buggy-but-conceptual / wrong-cond / degenerate)
_cases = [
    ("n is a palindrome (reads the same forwards and backwards)",
     "str(abs(n)) == str(abs(n)).reversed()"),          # buggy but intent clear -> YES
    ("n is a palindrome (reads the same forwards and backwards)",
     "n % 2 == 1"),                                       # a different condition -> NO
    ("n is a palindrome (reads the same forwards and backwards)",
     "s == s == d == e =="),                              # degenerate -> MALFORMED
]
for cond, stub in _cases:
    print(f"{judge_condition(cond, stub):9s}  <-  {stub!r}")

YES        <-  'str(abs(n)) == str(abs(n)).reversed()'
NO         <-  'n % 2 == 1'
MALFORMED  <-  's == s == d == e =='


In [41]:
# ── fold oracle K-way verdicts into the §6 demo generations ───────────────────
# Judges each generated stub (base / restrict / control from the demo cell) against ALL K
# conditions, then classifies the outcome relative to CURRENT/TARGET. 3×K oracle calls (cached).
# Requires the §6 demo cell (defines base/restr/ctrl/ex/CURRENT/TARGET/tvar) + §7 (oracle) to have run.
from rich.console import Group

_MARK = {"YES": "Y", "NO": "·", "MALFORMED": "M", "UNPARSED": "?"}
def _verdict_str(verdicts):
    cells = []
    for k, v in enumerate(verdicts):
        c = CUR_C if k == CURRENT else (TGT_C if k == TARGET else "grey58")
        cells.append(f"[{c}]{ex['var_names'][k]}={_MARK.get(v, '?')}[/]")
    return "  ".join(cells)

_rows = [
    f"[bold]oracle K-way verdicts[/]   Y=implements · =no  M=malformed   "
    f"(CURRENT=[{CUR_C}]slot{CURRENT} {cur.predicate_ids[CURRENT]}[/], "
    f"TARGET=[{TGT_C}]slot{TARGET} {tgt.predicate_ids[TARGET]}[/])",
]
for name, color, stub in [("baseline", "sea_green1", base),
                          (f"restrict {HEADSET}", "salmon1", restr),
                          ("control", "plum1", ctrl)]:
    v = judge_stub_kway(ex, stub)
    label, _yes = classify_stub(v, CURRENT, TARGET)
    _rows.append(f"  [bold]{name:18s}[/] {tvar} =[{color}]{stub!r}[/]")
    _rows.append(f"  {'':18s} {_verdict_str(v)}   →  [bold]{label}[/]")
rich_print(Group(*_rows))

2026-07-14 16:02:38 httpx INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-07-14 16:02:39 httpx INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-07-14 16:02:40 httpx INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-07-14 16:02:41 httpx INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-07-14 16:02:43 httpx INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-07-14 16:02:44 httpx INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-07-14 16:02:45 httpx INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-07-14 16:02:46 httpx INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-07-14 16:02:47 httpx INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-07-14

oracle K-way verdicts   Y=implements · =no  M=malformed   (CURRENT=slot2 num_divisors_gt, TARGET=slot4 
digit_sum_gt)
  baseline           c =' sum(1 for i in range(1, abs(n) + 1) if abs(n) % i == 0) > 6\n'
                     a=·  b=·  c=Y  d=·  e=·   →  CURRENT (no change)
  restrict goalscope c =' sum(int(digit) for digit in str(abs(n))) > 15\n'
                     a=·  b=·  c=·  d=·  e=Y   →  TARGET (flip)
  control            c =' len() > 6\n'
                     a=·  b=·  c=Y  d=·  e=·   →  CURRENT (no change)

## 8. Scaled experiment — N examples, one random (current, target) pair each

For each example: sample `K∈[4,6]`, a `current` slot (with ≥1 prior and ≥1 following) and one random
`target≠current`. Run the **clean/baseline** reference plus each head-set ∈ {gaze, goalscope, random},
scoring both the **soft** redirect (`p_target`, argmax-flip) and the **hard** readout (generate under the
restriction → oracle K-way judge → `classify_stub`). One flat record per (example, head-set); results are
checkpointed to a JSONL so the run is resumable.

Compute per example ≈ (1+3)×K soft forwards + (1+3) generations + (1+3)×K oracle calls (cached).

In [34]:
# ── config + per-example runner ───────────────────────────────────────────────
N_SAMPLES     = 50
K_RANGE       = (4, 6)                # inclusive n_conditions
HEADSET_NAMES = ["gaze", "goalscope", "random"]
WHICH_SPAN    = "full"               # restrict heads to the full spec span
MAX_NEW       = 32
BASE_SEED     = 1000
RUN_TAG       = "n50_singletarget_full"
CKPT = os.path.join(env_utils.DEFAULT_RESULTS_DIR, "dissociation",
                    MODEL_NAME.split("/")[-1], f"{RUN_TAG}.jsonl")
os.makedirs(os.path.dirname(CKPT), exist_ok=True)
print(f"checkpoint -> {CKPT}")


def _sample_pair(rng, K):
    current = rng.randint(1, K - 2)                        # need >=1 prior and >=1 following
    target  = rng.choice([k for k in range(K) if k != current])
    return current, target


def run_one_example(i):
    """Sample (K, current, target); run clean/baseline + each head-set (soft LL redirect +
    hard generation->oracle). Returns list of flat records (one per head-set + baseline)."""
    rng = random.Random(BASE_SEED + i)
    K = rng.randint(*K_RANGE)
    current, target = _sample_pair(rng, K)
    ex    = build_dissociation_prompt(mt, random.Random(7 * BASE_SEED + i),
                                      current_slot=current, n_conditions=K)
    spans = locate_condition_spans(mt, ex)
    refs  = get_condition_references(mt, ex)
    keep  = restrict_span_positions(spans, target, which=WHICH_SPAN)
    q_start = ex["pivot_pos"]

    ll_clean = score_condition_lls(mt, ex, refs)           # target-agnostic; computed once
    m_clean  = soft_metrics(ll_clean, current, target)

    common = dict(example_id=i, K=K, current=current, target=target,
                  direction=("fwd" if target > current else "bwd"), dist=abs(target - current),
                  current_pred=ex["spec"].predicate_ids[current],
                  target_pred=ex["spec"].predicate_ids[target],
                  p_target_clean=m_clean["p_target"], p_current_clean=m_clean["p_current"])

    def _rec(headset, m, sel):
        stub = generate_restricted_to_span(mt, ex["prompt_ids"], sel, keep, q_start, MAX_NEW)
        verd = judge_stub_kway(ex, stub)
        label, _ = classify_stub(verd, current, target)
        return dict(**common, headset=headset,
                    p_target=m["p_target"], p_current=m["p_current"],
                    dp_target=m["p_target"] - m_clean["p_target"],
                    argmax=m["argmax"], rank_target=m["rank_target"],
                    redirected_argmax=bool(m["redirected_argmax"]), margin=m["margin_tgt_cur"],
                    gen_stub=stub, verdicts=verd, oracle_label=label,
                    oracle_flip=bool(label.startswith("TARGET")))

    recs = [_rec("baseline", m_clean, [])]                 # no-mask reference (implements CURRENT)
    for hs in HEADSET_NAMES:
        ll = score_condition_lls(mt, ex, refs, selected_heads=HEAD_SETS[hs], keep_positions=keep)
        recs.append(_rec(hs, soft_metrics(ll, current, target), HEAD_SETS[hs]))
    return recs

checkpoint -> /disk/u/arnab/Codes/Projects/goals/results/dissociation/Qwen2.5-Coder-32B-Instruct/n50_singletarget_full.jsonl


In [35]:
# ── run (checkpointed / resumable) — re-run the cell to continue after an interrupt ──
done = set()
if os.path.exists(CKPT):
    with open(CKPT) as f:
        for line in f:
            done.add(json.loads(line)["example_id"])
logger.info(f"resuming: {len(done)}/{N_SAMPLES} examples already in {os.path.basename(CKPT)}")

for i in range(N_SAMPLES):
    if i in done:
        continue
    recs = run_one_example(i)
    with open(CKPT, "a") as f:
        for r in recs:
            f.write(json.dumps(r) + "\n")
    labs = {r["headset"]: r["oracle_label"].split(" ")[0] for r in recs if r["headset"] != "baseline"}
    logger.info(f"[{i+1:>2}/{N_SAMPLES}] K={recs[0]['K']} cur={recs[0]['current']}->tgt={recs[0]['target']} "
                f"({recs[0]['direction']} d{recs[0]['dist']}) :: {labs}")
print("done.")

2026-07-14 15:42:36 __main__ INFO     resuming: 0/50 examples already in n50_singletarget_full.jsonl
2026-07-14 15:42:39 httpx INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-07-14 15:42:40 httpx INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-07-14 15:42:41 httpx INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-07-14 15:42:42 httpx INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-07-14 15:42:43 httpx INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-07-14 15:42:48 httpx INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-07-14 15:42:49 httpx INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-07-14 15:42:50 httpx INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-07-14 15:42:51

KeyboardInterrupt: 

## 9. Aggregate + report

Load the checkpoint, then report per head-set: soft argmax-flip rate, oracle flip-rate, mean
`Δp_target`, and the oracle outcome breakdown (TARGET / CURRENT / OTHER / MALFORMED / NONE), plus the
soft↔hard agreement. The **money comparison** is gaze vs goalscope vs random: if a head-set redirects
(`TARGET (flip)`) far above the matched random baseline, its heads causally ground *which* sub-task is
implemented.

In [ ]:
import pandas as pd
from rich.table import Table

df = pd.read_json(CKPT, lines=True)
LABELS = ["TARGET", "CURRENT", "OTHER", "AMBIGUOUS", "MALFORMED", "NONE"]
def label_cat(l):
    for c in LABELS:
        if l.startswith(c):
            return c
    return "?"

iv = df[df.headset != "baseline"].copy()
iv["label_cat"] = iv.oracle_label.map(label_cat)

summary = iv.groupby("headset").agg(
    n=("headset", "size"),
    soft_flip=("redirected_argmax", "mean"),
    oracle_flip=("oracle_flip", "mean"),
    dp_target=("dp_target", "mean"),
    p_target=("p_target", "mean"),
    rank_target=("rank_target", "mean"),
).reindex(HEADSET_NAMES)
brk = (iv.groupby("headset")["label_cat"].value_counts(normalize=True)
         .unstack(fill_value=0).reindex(index=HEADSET_NAMES, columns=LABELS, fill_value=0))
agree = (iv.redirected_argmax == iv.oracle_flip).mean()
pb = np.corrcoef(iv.p_target, iv.oracle_flip.astype(float))[0, 1]

# baseline sanity: what does the no-mask generation implement?
base = df[df.headset == "baseline"]
base_current_rate = base.oracle_label.str.startswith("CURRENT").mean()

t = Table(title=f"redirect by head-set  (N={df.example_id.nunique()} examples, {len(iv)} interventions)")
for c, j in [("head-set","left"),("n","right"),("soft-flip","right"),("oracle-flip","right"),
             ("Δp_tgt","right"),("p_tgt","right"),("rank_t","right"),
             ("TARGET","right"),("CURRENT","right"),("OTHER","right"),("MALF","right"),("NONE","right")]:
    t.add_column(c, justify=j)
for hs in HEADSET_NAMES:
    s, b = summary.loc[hs], brk.loc[hs]
    style = "bold" if hs != "random" else "grey58"
    t.add_row(hs, str(int(s.n)), f"{s.soft_flip:.2f}", f"{s.oracle_flip:.2f}",
              f"{s.dp_target:+.2f}", f"{s.p_target:.2f}", f"{s.rank_target:.2f}",
              f"{b.TARGET:.2f}", f"{b.CURRENT:.2f}", f"{b.OTHER:.2f}", f"{b.MALFORMED:.2f}",
              f"{b.NONE:.2f}", style=style)
rich_print(t)
rich_print(f"baseline (no mask) lands on CURRENT: [bold]{base_current_rate:.2f}[/]  (sanity: should be high)")
rich_print(f"soft(argmax-flip) vs oracle-flip agreement = [bold]{agree:.3f}[/]   "
           f"point-biserial corr(p_target, oracle_flip) = [bold]{pb:.3f}[/]")

# by direction
bydir = iv.groupby(["headset", "direction"]).agg(
    n=("headset", "size"), soft_flip=("redirected_argmax", "mean"),
    oracle_flip=("oracle_flip", "mean"))
td = Table(title="flip-rate by head-set × direction (fwd = skip ahead, bwd = go back)")
for c in ["head-set", "dir", "n", "soft-flip", "oracle-flip"]:
    td.add_column(c, justify=("left" if c == "head-set" else "right"))
for hs in HEADSET_NAMES:
    for d in ["fwd", "bwd"]:
        if (hs, d) in bydir.index:
            r = bydir.loc[(hs, d)]
            td.add_row(hs, d, str(int(r.n)), f"{r.soft_flip:.2f}", f"{r.oracle_flip:.2f}")
rich_print(td)

In [ ]:
import matplotlib.pyplot as plt

COLORS     = {"gaze": "#1f77b4", "goalscope": "#d62728", "random": "#7f7f7f"}
LAB_COLORS = {"TARGET": "#55a868", "CURRENT": "#c44e52", "OTHER": "#8172b3",
              "AMBIGUOUS": "#ccb974", "MALFORMED": "#937860", "NONE": "#cccccc"}
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

# (1) money plot — flip-rate by head-set: soft (argmax) vs oracle
x, w = np.arange(len(HEADSET_NAMES)), 0.38
axes[0].bar(x - w/2, summary.soft_flip, w, label="soft (argmax-flip)", color="#4c72b0")
axes[0].bar(x + w/2, summary.oracle_flip, w, label="oracle-flip", color="#dd8452")
axes[0].set_xticks(x); axes[0].set_xticklabels(HEADSET_NAMES); axes[0].set_ylim(0, 1)
axes[0].set_ylabel("redirect (flip) rate"); axes[0].set_title("flip-rate by head-set"); axes[0].legend()

# (2) oracle outcome breakdown (stacked)
bottom = np.zeros(len(HEADSET_NAMES))
for lab in LABELS:
    axes[1].bar(HEADSET_NAMES, brk[lab], bottom=bottom, label=lab, color=LAB_COLORS[lab])
    bottom += brk[lab].values
axes[1].set_ylim(0, 1); axes[1].set_ylabel("fraction"); axes[1].set_title("oracle outcome breakdown")
axes[1].legend(fontsize=7, ncol=2)

# (3) oracle-flip rate vs |Δslot|
for hs in HEADSET_NAMES:
    g = iv[iv.headset == hs].groupby("dist")["oracle_flip"].mean()
    axes[2].plot(g.index, g.values, "o-", label=hs, color=COLORS[hs])
axes[2].set_xlabel("|Δslot|"); axes[2].set_ylabel("oracle-flip rate"); axes[2].set_ylim(0, 1)
axes[2].set_title("flip-rate vs distance"); axes[2].legend()

plt.tight_layout()
FIG = os.path.join(os.path.dirname(CKPT), f"{RUN_TAG}.png")
plt.savefig(FIG, dpi=110, bbox_inches="tight"); print("saved", FIG)
plt.show()